In [1]:
import pandas as pd

excel = pd.read_excel("Cartera2412.xlsx")
excel.to_csv("Cartera.xlsx")

In [66]:
tabla_cartera = pd.read_csv("Cartera.xlsx")
INSTRUMENTOS = {
    'Bonos Subordinados':'bonos_subordinados',
    'Bonos Financieros':'bonos_financieros',
    "Bonos":'bonos'
}


MONEDAS = {
    'Guaraníes': 155,
    'Dólares americanos':2
}

In [30]:
tabla_bonos = tabla_cartera[
    (pd.notna(tabla_cartera['Serie'])) &
    (tabla_cartera['Serie'] != 'nan') &
    (tabla_cartera['Instrumento'].isin(['Bonos Subordinados', 'Bonos Financieros', 'Bonos']))
]# se filtran solo los bonos de la tabla de cartera

In [4]:
#vamos a inicializar o filtrar los numeros de serie y comitentes para facilitar la select de datos
series = {}

for indice in tabla_bonos.index:
    fila = tabla_bonos.loc[indice] # se inicializa el registro(fila de la tabla)
    if fila['Serie'] in series.keys():# 
        if fila['Casa de bolsa '] in series[fila['Serie']]: #se valida si ya existe esta serie con el mismo comitente (casa de bolsa)
            pass
        else:
            if pd.notna(fila['Casa de bolsa ']) or str(fila['Casa de bolsa ']) != 'nan' and str(fila['Casa de bolsa ']) != 'None':
                series[fila['Serie']].append(fila['Casa de bolsa '])
    else:
        series[fila['Serie']] = []
    



In [5]:
def obtener_fecha_compra(registros):
    for indice in registros.index:
        if pd.notna(registros.loc[indice]['Fecha Compra']) and str(registros.loc[indice]['Fecha Compra']) != 'nan' and str(fila['Casa de bolsa ']) != 'None':
            return str(registros.loc[indice]['Fecha Compra'])


def obtener_fecha_vencimiento_max(capitales):

    return str(capitales['Fecha Vencimiento Serie'].max()).split(' ')[0]



def validar_registro(registros):
    """Valida que los registros cumplan con los requisitos:
       - Que tenga fecha de compra
       - Que tenga fecha de vencimiento
       - Que existan registros de vencimientos y capital
    """
    valido = True
    registros = registros.loc[registros['Tipo'] == 'Capital']   
    # Validar fecha_compra
    if obtener_fecha_compra(registros) is None or str(obtener_fecha_compra(registros)) == 'nan' or pd.isna(obtener_fecha_compra(registros)):
        valido = False

    # Validar fecha_vencimiento
    if obtener_fecha_vencimiento_max(registros) is None or str(obtener_fecha_vencimiento_max(registros)) == 'nan' or pd.isna(obtener_fecha_vencimiento_max(registros)):
        valido = False

    return valido

def obtener_datos_de_cabecera(capital):
    """Obtiene los datos para la cabecera o registro principla"""
    calificacion = ''
    tasa_interes = 0
    instrumento = None  # Definir por defecto
    moneda = None
    emisor = None
    comitente = None
    for index in capital.index:
        if capital.loc[index]['Calificacion de riesgo']:
            calificacion = capital.loc[index]['Calificacion de riesgo']
            # Limpieza de la tasa de interés
            tasa_interes_str = str(capital.loc[index]['Tasa Interes'])  # Convertir a string
            tasa_interes_str = tasa_interes_str.split('\\')[0]  # Quitar caracteres extraños
            tasa_interes_str = tasa_interes_str.replace(',', '.')  # Cambiar coma por punto
            tasa_interes_str = tasa_interes_str.replace('\xa0', '')  # Eliminar espacios invisibles
            tasa_interes_str = tasa_interes_str.replace('%', '')  # Eliminar el símbolo de porcentaje
            try:
                tasa_interes = float(tasa_interes_str)  # Convertir a float
            except ValueError:
                print(f"Error al convertir '{tasa_interes_str}' a float")  # Depuración opcional
                tasa_interes = None  # O manejar el error de otra forma

            instrumento = capital.loc[index]['Instrumento']
            moneda = capital.loc[index]['Moneda']
            emisor = capital.loc[index]['Emisor']
            comitente = capital.loc[index]['Comitente']
    return calificacion, tasa_interes,instrumento, moneda, emisor, comitente


In [6]:
registros_para_bonos = {}


contador_registros = 0
# se iteran los registros y se procesan los datos que se van a pasar al odoo
for serie in series:
    # se iteran las series y la casa de bolsa
    for casa_bolsa in series[serie]:
        # se inicializan solos los registros relacionados con la cartera que vamos a procesar
        registros = tabla_bonos.loc[(tabla_bonos['Serie']==serie) & (tabla_bonos['Casa de bolsa ']==casa_bolsa)]
        registros['Importe Valorizado'] = pd.to_numeric(registros['Importe Valorizado'], errors='coerce')

        # dependiendo de la cartera tiene solo un registro de capital o varios
        # de ese o esos registros podemos obtener el importe valorizado
        # Lo demas son los intereses, tambien se suman obteniendo los intereses, aunque ahora hay un campo computado para calcular
        capitales = registros.loc[registros['Tipo'] == 'Capital']
        intereses = registros.loc[registros['Tipo'] == 'Intereses']
        calificacion, tasa_intereses, instrumento, moneda, emisor, comitente = obtener_datos_de_cabecera(capitales)
        if validar_registro(registros):# logear despues los que estan facllando
            
            cabecera = {
                'serie':serie,
                'fecha_compra':obtener_fecha_compra(registros),
                'fecha_vencimiento': obtener_fecha_vencimiento_max(capitales),
                'calificacion_riesgo': calificacion,
                'tasa_interes': tasa_intereses,
                'importe_valorizado':capitales['Importe Valorizado'].sum(),
                'instrumento':INSTRUMENTOS[instrumento],
                'moneda':MONEDAS[moneda],
                'emision': emisor,
                'partner_id': False,
                'comitente':comitente,
                'vencimietos': []
                
            }
            # si el capital se cobra fraccionado 
            if len(capitales.index) > 1:
                for indice in capitales.index:
                    vencimiento = {
                        'name':serie,
                        'fecha_vencimiento': capitales.loc[indice]['Fecha Vencimiento Serie'],
                        'state': ESTADOS[capitales.loc[indice]['Estado']],
                        'amortizacion': 'pagocap',
                        'total': capitales.loc[indice]['Importe Valorizado']
                    }
                    cabecera['vencimietos'].append(vencimiento)
            else:
                for indice in capitales.index:
                    vencimiento = {
                        'name':serie,ntre el lunes al ju
                        'fecha_vencimiento': capitales.loc[indice]['Fecha Vencimiento Serie'],
                        'state': ESTADOS[capitales.loc[indice]['Estado']],
                        'amortizacion': 'pagocap',
                        'total': capitales.loc[indice]['Importe Valorizado']
                    }
                    cabecera['vencimietos'].append(vencimiento)
            # cobro de intereses
            for indice in intereses.index:
                vencimiento = {
                    'fecha_vencimiento': intereses.loc[indice]['Fecha Vencimiento Serie'],
                    'interes_titulo':0,
                    'estado': ESTADOS[intereses.loc[indice]['Estado']],
                    'tipo':'vtoInt'
                }
                cabecera['vencimietos'].append(vencimiento)
                
                
                

        
        
    

SyntaxError: ':' expected after dictionary key (373995942.py, line 50)

In [ ]:
tabla_cartera.loc[tabla_cartera['Serie']=='PYCGS01F8629']['Comitente']

In [7]:
reg = tabla_bonos.loc[(tabla_bonos['Serie']=='PYSUD01F4473')]
reg.to_csv("Test.csv")

In [8]:
casas = tabla_cartera['Emisor']

In [9]:
casas = tabla_cartera['Emisor']

In [10]:
list_casa = []
for casa in casas:
    if casa not in list_casa:
        list_casa.append(casa)

In [28]:
for i in list_casa:
    print(i)

SUDAMERIS BANK S.A.E.C.A.
BANCO BASA S.A.
CEMENTOS CONCEPCIÓN SOCIEDAD ANÓNIMA EMISORA
BANCO RIO SAECA
RIEDER & CÍA. S.A.C.I.
SOLAR BANCO S.A.E.
TU FINANCIERA SAECA
ENEX PARAGUAY S.A.E.
UENO HOLDING S.A.E.C.A.
FINANCIERA PARAGUAYO JAPONESA S.A.E.C.A.
MINISTERIO DE HACIENDA
BANCO FAMILIAR S.A.E.C.A.
FIC DE FINANZAS
VISION BANCO S.A.E.C.A.
ZETA BANCO S.A.E.C.A.
BANCO CONTINENTAL S.A.E.C.A.
BANCO BILBAO VIZCAYA ARGENTARIA PARAGUAY S.A.
CORPORACION INTERAMERICANA DE INVERSIONES
TAPE PORA S.A.E.
BANCO ATLAS S.A.
FONDO MUTUO BASA
IZAGUIRRE BARRAIL INVERSORA S.A.E.C.A.
VICENTE SCAVONE & CÍA. S.A.E.
TELEFONICA CELULAR DEL PARAGUAY S.A.E.
MUNICIPALIDAD DE FERNANDO DE LA MORA
TAPE RUVICHA S.A.E.C.A.
AGENCIA FINANCIERA DE DESARROLLO 
BANCO ITAÚ PARAGUAY S.A.
IMPERIAL COMPAÑÍA DISTRIBUIDORA DE PETRÓLEO Y DERIVADOS S.A.E.
GAS CORONA S.A.E.C.A.
HILAGRO 
BANCOP S.A.
GRUPO INTERNACIONAL DE FINANZAS S.A.E.C.A.
IMPORT CENTER S.A.
ELECTROBAN S.A.E.C.A.
KUROSU & CÍA. S.A.
FIDEICOMISO LA SUSANA
NUCLEO S.A.

In [78]:
tabla_fondos = tabla_cartera.loc[tabla_cartera['Instrumento']=='Fondos']
print(tabla_cartera.loc[tabla_cartera['Instrumento']=='Fondo'].index)
for indice in tabla_fondos.index:

    registro = tabla_fondos.loc[indice]
    
    tasa_interes_str = str(tabla_fondos.loc[indice]['Tasa Interes'])  # Convertir a string
    tasa_interes_str = tasa_interes_str.split('\\')[0]  # Quitar caracteres extraños
    tasa_interes_str = tasa_interes_str.replace(',', '.')  # Cambiar coma por punto
    tasa_interes_str = tasa_interes_str.replace('\xa0', '')  # Eliminar espacios invisibles
    tasa_interes_str = tasa_interes_str.replace('%', '')  # Eliminar el símbolo de porcentaje
    try:
        tasa_interes = float(tasa_interes_str)  # Convertir a float
    except ValueError:
        print(f"Error al convertir '{tasa_interes_str}' a float")
        tasa_interes = None
    #calificacion, tasa_intereses, instrumento, moneda, emisor, comitente = obtener_datos_de_cabecera(registro)
    
    cabecera = {
                'serie':registro['Serie'],
                'fecha_compra':False,
                'fecha_vencimiento': False,
                'calificacion_riesgo': registro['Calificacion de riesgo'],
                'tasa_interes': tasa_interes_str,
                'importe_valorizado':registro['Importe Valorizado'],
                'instrumento':'fondos',
                'moneda':MONEDAS[registro['Moneda']],
                'emision': registro['Emisor'],
                'partner_id': False,
                'comitente':False,
                'vencimietos': {
                    'fecha_vencimiento': registro['Fecha Vencimiento Serie'],
                    'interes_titulo':0,
                    'estado': 'pendiente',
                    'tipo':'vtoInt',
                    'total': registro['Importe Valorizado']
                }
                
            }
    print(cabecera)
    
    

Index([], dtype='int64')
{'serie': 'Fondos', 'fecha_compra': False, 'fecha_vencimiento': False, 'calificacion_riesgo': 'AA + f Estable', 'tasa_interes': '4.50', 'importe_valorizado': ' - ', 'instrumento': 'fondos', 'moneda': 155, 'emision': 'FONDO MUTUO BASA', 'partner_id': False, 'comitente': False, 'vencimietos': {'fecha_vencimiento': 'No aplica | Fondos Mutuos', 'interes_titulo': 0, 'estado': 'pendiente', 'tipo': 'vtoInt', 'total': ' - '}}
{'serie': 'Fondos', 'fecha_compra': False, 'fecha_vencimiento': False, 'calificacion_riesgo': 'AA + f Estable', 'tasa_interes': '2.50', 'importe_valorizado': '317.7', 'instrumento': 'fondos', 'moneda': 2, 'emision': 'FONDO MUTUO BASA', 'partner_id': False, 'comitente': False, 'vencimietos': {'fecha_vencimiento': 'No aplica | Fondos Mutuos', 'interes_titulo': 0, 'estado': 'pendiente', 'tipo': 'vtoInt', 'total': '317.7'}}
{'serie': 'Fondos', 'fecha_compra': False, 'fecha_vencimiento': False, 'calificacion_riesgo': 'AA + f Estable', 'tasa_interes': '

154              Guaraníes
156     Dólares americanos
1236             Guaraníes
1346    Dólares americanos
1594    Dólares americanos
3060             Guaraníes
3061    Dólares americanos
3517             Guaraníes
3543             Guaraníes
3600    Dólares americanos
3633             Guaraníes
3634    Dólares americanos
3654             Guaraníes
3655    Dólares americanos
Name: Moneda, dtype: object